# Superpixel GATv2 + hybrid CNN — chest X-ray classification

**Architecture:** frozen ResNet18 features → SLIC superpixel graph → GATv2 → mean+max pooling → MLP
**Task:** 5-class chest X-ray (Cardiac, ChronicLung, Normal, Pleural, TB)
**Dataset:** <https://www.kaggle.com/datasets/shakib0hasan/capstone-c-dataset>

This notebook is the *driver*. All logic lives in the `cxr_gnn` package
([repository](https://github.com/isratjahan829/Superpixel-GATv2-HybridCNN-CXR)), so the code you run here is the code that is
tested and reviewed — the notebook cannot drift away from it.

### What this run produces

| Step | Output |
|---|---|
| 1–6 | Data discovery, image-level split, graph cache, training |
| 7 | Held-out test evaluation + 7 figures |
| 8 | Stratified 5-fold cross-validation |
| 9 | Ablation: feature arms **and** architecture arms |
| 10 | Baselines (GCN / GraphSAGE / GAT / end-to-end ResNet18) + full statistical suite |
| 11 | Split-ratio sensitivity (5 configs × 5 seeds) + error analysis |
| 12 | Conformal prediction (LAC / APS / RAPS / class-conditional LAC) |
| 13 | Calibration (ECE / MCE + temperature scaling) |
| 14 | Robustness under input perturbation |
| 15 | Every evaluation table, rebuilt from the saved artifacts |

### Runtime

On a Kaggle T4 with the real dataset the full run takes roughly 1.5–2 hours,
dominated by the end-to-end ResNet18 baseline. Set `FAST = True` in the config
cell for a few-minute smoke test (results from a fast run are **not** reportable).

### Notes on correctness

The pipeline splits on raw images *before* augmentation, measures train and
validation metrics in the same `eval()` mode, and reuses one set of fold indices
across every model so the paired McNemar/DeLong tests are valid. See
`docs/EVALUATION_TABLES_REVIEW.md` for the specific defects this rewrite fixes.

## Step 0 — Environment

In [1]:
# Make the cxr_gnn package importable.
#   - Running inside a clone of the repository: nothing to do.
#   - Kaggle/Colab with internet on: install straight from GitHub.
#   - No internet: add the repo as a Kaggle dataset and point PACKAGE_PATH at it.
import importlib, os, subprocess, sys

PACKAGE_PATH = None          # e.g. "/kaggle/input/cxr-gnn-source"

for candidate in [PACKAGE_PATH, ".", "..", "/kaggle/working/Superpixel-GATv2-HybridCNN-CXR"]:
    if candidate and os.path.isdir(os.path.join(candidate, "cxr_gnn")):
        sys.path.insert(0, os.path.abspath(candidate))
        break

try:
    import cxr_gnn                                     # noqa: F401
    print("cxr_gnn found at", os.path.dirname(cxr_gnn.__file__))
except ImportError:
    print("cxr_gnn not found locally -- installing from GitHub ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/isratjahan829/Superpixel-GATv2-HybridCNN-CXR.git"], check=True)
    importlib.invalidate_caches()
    import cxr_gnn
    print("installed:", os.path.dirname(cxr_gnn.__file__))

cxr_gnn found at /home/user/Superpixel-GATv2-HybridCNN-CXR/cxr_gnn


In [2]:
# torch-geometric is the only dependency Kaggle images do not always ship.
# torch-scatter / torch-sparse are deliberately NOT installed: PyG 2.x does not
# need them here, and pip builds them from source (slow, and often fails).
try:
    import torch_geometric
    print("torch-geometric", torch_geometric.__version__)
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"], check=True)
    import torch_geometric
    print("torch-geometric", torch_geometric.__version__, "(just installed)")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch-geometric 2.8.0.post1


## Step 1 — Configuration

`Config` is a frozen dataclass: it cannot be mutated after construction, so
there is never any doubt about which settings a given cell ran under. Derive a
variant explicitly with `cfg.replace(...)`.

The primary split is **S4 (70/20/10)** — the configuration the split-ratio
sensitivity analysis selects — and every reported number uses it.

In [3]:
import os
import numpy as np
import torch

from cxr_gnn.config import Config, CLEAN_LABELS, CLASS2IDX, IDX2CLASS, NUM_CLASSES, DISPLAY_NAMES
from cxr_gnn.utils import set_seed, get_device, setup_logging, get_logger

# FAST=True shrinks every budget so the whole notebook runs in minutes.
# Use it to check the pipeline works; do not report numbers from a fast run.
FAST = not os.path.isdir("/kaggle/input")

DATA_ROOT = os.environ.get("CXR_DATA_ROOT")   # None -> auto-detect under /kaggle/input
WORK_DIR = os.environ.get("CXR_WORK_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else "outputs")

kw = {}
if DATA_ROOT:
    kw["data_root"] = DATA_ROOT
if FAST:
    # Smaller images and fewer superpixels as well as fewer epochs: on CPU the
    # graph build dominates, not the training.
    kw.update(img_size=128, n_segments=120,
              epochs=8, cv_epochs=8, cv_patience=3,
              sensitivity_epochs=6, sensitivity_patience=3,
              sensitivity_seeds=(42, 43),
              cnn_epochs=2, cnn_patience=2, n_bootstrap=300)

cfg = Config(work_dir=WORK_DIR, **kw)
device = get_device()
logger = setup_logging(cfg.work_dir)
set_seed(cfg.seed)

TARGET_NAMES = [DISPLAY_NAMES.get(IDX2CLASS[i], IDX2CLASS[i]) for i in range(NUM_CLASSES)]

print(f"Mode        : {'FAST (smoke test -- not reportable)' if FAST else 'FULL'}")
print(f"Device      : {device}")
print(f"Work dir    : {cfg.work_dir}")
print(f"Classes     : {CLEAN_LABELS}")
print(f"Node feats  : {cfg.node_feat_dim} (deep={cfg.deep_feat_dim} + hand={cfg.hand_feat_dim})")
print(f"Edge feats  : {cfg.edge_feat_dim}")
print(f"Primary split: train={1 - cfg.test_size - cfg.val_size:.0%} "
      f"val={cfg.val_size:.0%} test={cfg.test_size:.0%}  (S4)")
print(f"Bootstrap   : B={cfg.n_bootstrap}")

Mode        : FAST (smoke test -- not reportable)
Device      : cpu
Work dir    : outputs
Classes     : ['Cardiac', 'ChronicLung', 'Normal', 'Pleural', 'TB']
Node feats  : 140 (deep=128 + hand=12)
Edge feats  : 2
Primary split: train=70% val=20% test=10%  (S4)
Bootstrap   : B=300


## Step 2 — Data discovery and image-level split

The split happens on **raw images, before augmentation**. Building all the
graphs first and splitting afterwards puts augmented copies of training images
into val/test, which leaks and inflates every metric downstream.

If the Kaggle dataset is not mounted, the cell falls back to a small synthetic
phantom dataset so the notebook still runs end to end. Phantom images are
procedural, not radiographs — metrics computed on them mean nothing beyond
"the code executes".

In [4]:
from cxr_gnn.data.dataset import find_data_root, collect_samples, stratified_image_split

try:
    data_root = find_data_root(cfg)
    USING_SYNTHETIC = False
except FileNotFoundError:
    print("Real dataset not found -- generating synthetic phantoms for a runnable demo.\n")
    from tools.make_demo_dataset import generate      # noqa: E402
    generate("demo_data", per_class=20 if FAST else 60, size=cfg.img_size, seed=0)
    cfg = cfg.replace(data_root=os.path.abspath("demo_data"))
    data_root = find_data_root(cfg)
    USING_SYNTHETIC = True

samples = collect_samples(data_root, cfg)
splits = stratified_image_split(samples, cfg, cfg.seed)

from cxr_gnn.utils import write_run_metadata
meta = write_run_metadata(cfg.work_dir, cfg, device, len(samples), data_root,
                          synthetic=USING_SYNTHETIC)

print(f"\nTotal images : {len(samples)}")
for k in ("train", "val", "test"):
    print(f"{k.capitalize():6s} images: {len(splits[k])}")
if USING_SYNTHETIC:
    print("\n*** SYNTHETIC DATA -- results below are a smoke test, not findings. ***")

Real dataset not found -- generating synthetic phantoms for a runnable demo.



2026-07-25 07:55:52 [INFO] cxr_gnn.data.dataset: Found data root: /home/user/Superpixel-GATv2-HybridCNN-CXR/demo_data


2026-07-25 07:55:52 [INFO] cxr_gnn.data.dataset: Found class folders: ['Cardiac Pathology', 'Cronic Lung Disease', 'Normal', 'TB', 'plural Pathology']


2026-07-25 07:55:52 [INFO] cxr_gnn.data.dataset: Images per folder:


2026-07-25 07:55:52 [INFO] cxr_gnn.data.dataset:   Cardiac Pathology            -> Cardiac      : 20


...

2026-07-25 07:55:52 [INFO] cxr_gnn.data.dataset: Split [train]: total=57  {'Cardiac': 14, 'ChronicLung': 5, 'Normal': 19, 'Pleural': 5, 'TB': 14}


2026-07-25 07:55:52 [INFO] cxr_gnn.data.dataset: Split [val]: total=15  {'Cardiac': 4, 'ChronicLung': 1, 'Normal': 5, 'Pleural': 1, 'TB': 4}


2026-07-25 07:55:52 [INFO] cxr_gnn.data.dataset: Split [test]: total=9  {'Cardiac': 2, 'ChronicLung': 1, 'Normal': 3, 'Pleural': 1, 'TB': 2}


2026-07-25 07:55:52 [WARNING] cxr_gnn.utils: This run is NOT reportable (synthetic data and reduced budgets). Its numbers verify that the pipeline executes; they are not findings.



Total images : 81
Train  images: 57
Val    images: 15
Test   images: 9

*** SYNTHETIC DATA -- results below are a smoke test, not findings. ***


## Step 3 — Frozen ResNet18 encoder

Stem + layer1 + layer2 only, giving a 128-channel map at 1/8 resolution. Every
parameter is frozen, so the encoder contributes **zero trainable parameters** to
the hybrid model — the reason the GATv2 hybrid is two orders of magnitude
smaller than a fine-tuned ResNet18.

In [5]:
from cxr_gnn.models.encoder import build_encoder

encoder = build_encoder(device)
frozen = sum(p.numel() for p in encoder.parameters())
print(f"Encoder: {frozen:,} frozen parameters (0 trainable)")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


2026-07-25 07:55:53 [WARNING] cxr_gnn.models.encoder: Pretrained weights unavailable (<urlopen error Tunnel connection failed: 403 Forbidden>). Using random init.


2026-07-25 07:55:53 [WARNING] cxr_gnn.models.encoder: Falling back to randomly initialised ResNet18.


2026-07-25 07:55:53 [INFO] cxr_gnn.models.encoder: Encoder frozen: 683072 params (not trained)


Encoder: 683,072 frozen parameters (0 trainable)


## Step 4 — Build or load the graph cache

Each image becomes a SLIC superpixel graph: ~180 nodes carrying 128 deep
features pooled from the encoder plus 12 hand-crafted descriptors, with edges
between adjacent regions. This is the slow step; the result is cached and
reloaded on subsequent runs. The cache stores a signature of every
graph-affecting config field and refuses to load if any of them changed.

In [6]:
from cxr_gnn.data.cache import build_or_load_cache

train_graphs, val_graphs, test_graphs, kept_items = build_or_load_cache(
    splits, cfg, encoder, device, cfg.seed)

nfd = train_graphs[0].x.shape[1]
efd = train_graphs[0].edge_attr.shape[1] if train_graphs[0].edge_attr is not None else None
n_aug = sum(int(getattr(g, "is_aug", 0)) for g in train_graphs)

print(f"\nGraphs -> train {len(train_graphs)} (of which {n_aug} augmented) | "
      f"val {len(val_graphs)} | test {len(test_graphs)}")
print(f"Node feature dim: {nfd} | edge feature dim: {efd}")
print(f"Mean nodes per graph: {np.mean([g.num_nodes for g in train_graphs]):.0f}")
print(f"Path-aligned kept items: " +
      ", ".join(f"{k}={len(v)}" for k, v in kept_items.items()))

2026-07-25 07:55:53 [INFO] cxr_gnn.data.cache: Building graph cache (first run -- a few minutes on GPU) ...


2026-07-25 07:56:12 [INFO] cxr_gnn.data.cache: Augmentation target per class: 19 | originals: {'Cardiac': 14, 'ChronicLung': 5, 'Normal': 19, 'Pleural': 5, 'TB': 14}


2026-07-25 07:56:21 [INFO] cxr_gnn.data.cache: Built: train=95 (orig=57) val=15 test=9


2026-07-25 07:56:21 [INFO] cxr_gnn.data.cache: Cache saved -> outputs/graph_cache.pt



Graphs -> train 95 (of which 38 augmented) | val 15 | test 9
Node feature dim: 140 | edge feature dim: 2
Mean nodes per graph: 121
Path-aligned kept items: train=57, val=15, test=9


## Step 5 — Dataloaders and pooled index sets

`WeightedRandomSampler` gives the minority classes (ChronicLung, Pleural)
roughly equal exposure per epoch; with plain shuffling they are invisible in
most batches.

The pools built here are reused by *every* later analysis. They contain only
original graphs — an augmented graph inside a held-out fold would inflate all
the cross-validation numbers.

In [7]:
from torch_geometric.loader import DataLoader
from cxr_gnn.training.trainer import make_weighted_loader, make_loss_weights

train_loader = make_weighted_loader(train_graphs, cfg.batch_size, NUM_CLASSES, seed=cfg.seed)
val_loader = DataLoader(val_graphs, cfg.batch_size, shuffle=False)
test_loader = DataLoader(test_graphs, cfg.batch_size, shuffle=False)

orig_graphs = [g for g in train_graphs if int(getattr(g, "is_aug", 0)) == 0] + val_graphs
orig_labels = np.array([int(g.y) for g in orig_graphs])
orig_items = ([(p, CLASS2IDX[c]) for p, c in kept_items["train"]]
              + [(p, CLASS2IDX[c]) for p, c in kept_items["val"]])
assert len(orig_items) == len(orig_graphs), "cache is stale -- rebuild it"

all_graphs = orig_graphs + test_graphs
all_labels = np.array([int(g.y) for g in all_graphs])

print(f"Batches -> train {len(train_loader)} | val {len(val_loader)} | test {len(test_loader)}")
print(f"CV pool (originals only): {len(orig_graphs)} graphs")
print(f"Sensitivity pool (all):   {len(all_graphs)} graphs")

Batches -> train 3 | val 1 | test 1
CV pool (originals only): 72 graphs
Sensitivity pool (all):   81 graphs


## Step 6 — Train the GATv2 classifier

`optimize_epoch()` does the gradient updates with dropout and DropEdge active;
`evaluate()` measures **both** splits in `eval()` mode. Measuring training
metrics during the optimisation pass (dropout on) is what made the original
notebook's train accuracy look lower than its validation accuracy.

In [8]:
from cxr_gnn.models.gatv2 import GATv2Classifier
from cxr_gnn.training.trainer import Trainer

model = GATv2Classifier(nfd, efd, cfg).to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(model)

loss_weights = make_loss_weights(train_graphs, NUM_CLASSES, device)
trainer = Trainer(model, cfg, loss_weights, device)
history = trainer.fit(train_loader, val_loader, cfg.ckpt_file)
trainer.load_best(cfg.ckpt_file)
print("\nTraining finished; best checkpoint reloaded.")

Trainable parameters: 79,293
GATv2Classifier(
  (in_bn): BatchNorm1d(140, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (gat1): GATv2Conv(140, 48, heads=4)
  (bn1): BatchNorm1d(192, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (gat2): GATv2Conv(192, 48, heads=1)
  (bn2): BatchNorm1d(48, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (head): Sequential(
    (0): Linear(in_features=96, out_features=48, bias=True)
    (1): ELU(alpha=1.0)
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=48, out_features=5, bias=True)
  )
)


2026-07-25 07:56:22 [INFO] cxr_gnn.training.trainer: E001 | tr 1.633/0.221 | va 1.602/0.333 bacc 0.200 | lr 3.0e-04  <- best


2026-07-25 07:56:22 [INFO] cxr_gnn.training.trainer: E002 | tr 1.619/0.368 | va 1.598/0.400 bacc 0.400 | lr 3.0e-04  <- best


2026-07-25 07:56:23 [INFO] cxr_gnn.training.trainer: E003 | tr 1.574/0.453 | va 1.591/0.400 bacc 0.400 | lr 3.0e-04  <- best


2026-07-25 07:56:23 [INFO] cxr_gnn.training.trainer: E004 | tr 1.571/0.232 | va 1.582/0.133 bacc 0.240 | lr 3.0e-04  <- best


...

2026-07-25 07:56:25 [INFO] cxr_gnn.training.trainer: E007 | tr 1.444/0.526 | va 1.520/0.400 bacc 0.600 | lr 3.0e-04  <- best


2026-07-25 07:56:25 [INFO] cxr_gnn.training.trainer: E008 | tr 1.463/0.442 | va 1.493/0.400 bacc 0.600 | lr 3.0e-04  <- best


2026-07-25 07:56:25 [INFO] cxr_gnn.training.trainer: Best val loss: 1.4935 (3.4 s)


2026-07-25 07:56:25 [INFO] cxr_gnn.training.trainer: Loaded best checkpoint from outputs/best_gatv2.pt



Training finished; best checkpoint reloaded.


## Step 7 — Held-out test evaluation

In [9]:
import json
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             matthews_corrcoef, classification_report)
from train import predict_all
from cxr_gnn.evaluation.stats import macro_auc

te_y, te_p, te_pr = predict_all(model, test_loader, device)
tr_y, tr_p, _ = predict_all(model, train_loader, device)

test_metrics = {
    "accuracy": float(accuracy_score(te_y, te_p)),
    "balanced_accuracy": float(balanced_accuracy_score(te_y, te_p)),
    "macro_f1": float(f1_score(te_y, te_p, average="macro")),
    "macro_auc": float(macro_auc(te_y, te_pr, NUM_CLASSES)),
    "mcc": float(matthews_corrcoef(te_y, te_p)),
    "train_accuracy_eval_mode": float(accuracy_score(tr_y, tr_p)),
    "n_total": len(samples),
    "n_test": int(len(te_y)),
}
for k, v in test_metrics.items():
    print(f"{k:26s}: {v:.4f}" if isinstance(v, float) else f"{k:26s}: {v}")
print()
print(classification_report(te_y, te_p, labels=list(range(NUM_CLASSES)),
                            target_names=TARGET_NAMES, zero_division=0))

with open(f"{cfg.work_dir}/test_results.json", "w") as f:
    json.dump(test_metrics, f, indent=2)
print(f"Saved -> {cfg.work_dir}/test_results.json")

accuracy                  : 0.4444
balanced_accuracy         : 0.6000
macro_f1                  : 0.4889
macro_auc                 : 0.9429
mcc                       : 0.4637
train_accuracy_eval_mode  : 0.5368
n_total                   : 81
n_test                    : 9

                      precision    recall  f1-score   support

   Cardiac Pathology       0.29      1.00      0.44         2
Chronic Lung Disease       1.00      1.00      1.00         1
              Normal       0.00      0.00      0.00         3
   Pleural Pathology       1.00      1.00      1.00         1
   Tuberculosis (TB)       0.00      0.00      0.00         2

            accuracy                           0.44         9
           macro avg       0.46      0.60      0.49         9
        weighted avg       0.29      0.44      0.32         9

Saved -> outputs/test_results.json


### Figures

Note that `plot_attention_saliency` takes the **encoder**. Passing `None`
produces 12-feature graphs for a model expecting 140, so every forward pass
raises and the figure comes out blank.

In [10]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from cxr_gnn.evaluation import visualization as viz

wd = cfg.work_dir
viz.plot_training_curves(history, wd)
viz.plot_confusion(te_y, te_p, TARGET_NAMES, wd)
viz.plot_per_class_metrics(te_y, te_p, TARGET_NAMES, wd)
viz.plot_roc_pr(te_y, te_pr, TARGET_NAMES, wd)
viz.plot_tsne(model, train_loader, test_loader, device, wd, TARGET_NAMES)
viz.plot_superpixel_graphs(samples, cfg, TARGET_NAMES, wd)
viz.plot_attention_saliency(model, samples, cfg, TARGET_NAMES, encoder, device, wd)

for path in sorted(glob.glob(f"{wd}/fig[1-7]*.png")):
    print(f"\n--- {os.path.basename(path)} ---")
    plt.figure(figsize=(14, 6))
    plt.imshow(mpimg.imread(path))
    plt.axis("off")
    plt.tight_layout()
    plt.show()


--- fig1_training_curves.png ---

--- fig2_confusion.png ---

--- fig3_per_class_metrics.png ---

--- fig4_roc_pr.png ---

--- fig5_tsne.png ---

--- fig6_superpixel_graphs.png ---



--- fig7_attention_saliency.png ---


## Step 8 — Stratified 5-fold cross-validation

Original images only. MCC is reported per fold alongside accuracy/F1/AUC: with
this class imbalance it is the number worth quoting.

In [11]:
from cxr_gnn.training.crossval import run_cross_validation

cv_results = run_cross_validation(orig_graphs, orig_labels, cfg, device, wd)

print("\n5-fold CV summary:")
for k in ["Accuracy", "Balanced-Acc", "Macro-F1", "Macro-AUC", "MCC"]:
    print(f"  {k:14s}: {cv_results[k]['mean']:.4f} +/- {cv_results[k]['std']:.4f}")

2026-07-25 07:56:41 [INFO] cxr_gnn.training.crossval: Fold 1: acc 0.1333 | bal-acc 0.4000 | F1 0.2267 | AUC 0.8149 | MCC 0.2200 | n(tr/va/te)=48/9/15


2026-07-25 07:56:42 [INFO] cxr_gnn.training.crossval: Fold 2: acc 0.1333 | bal-acc 0.4000 | F1 0.2267 | AUC 0.7713 | MCC 0.2200 | n(tr/va/te)=48/9/15


2026-07-25 07:56:43 [INFO] cxr_gnn.training.crossval: Fold 3: acc 0.1429 | bal-acc 0.4000 | F1 0.2286 | AUC 0.8533 | MCC 0.2288 | n(tr/va/te)=49/9/14


2026-07-25 07:56:44 [INFO] cxr_gnn.training.crossval: Fold 4: acc 0.2857 | bal-acc 0.4000 | F1 0.2571 | AUC 0.7795 | MCC 0.3257 | n(tr/va/te)=49/9/14


2026-07-25 07:56:45 [INFO] cxr_gnn.training.crossval: Fold 5: acc 0.0714 | bal-acc 0.2000 | F1 0.0267 | AUC 0.8818 | MCC 0.0000 | n(tr/va/te)=49/9/14


...

2026-07-25 07:56:45 [INFO] cxr_gnn.training.crossval:   Macro-F1      : 0.1931 +/- 0.0840


2026-07-25 07:56:45 [INFO] cxr_gnn.training.crossval:   Macro-AUC     : 0.8202 +/- 0.0424


2026-07-25 07:56:45 [INFO] cxr_gnn.training.crossval:   MCC           : 0.1989 +/- 0.1072


2026-07-25 07:56:45 [INFO] cxr_gnn.training.crossval: Saved outputs/cv_results.json



5-fold CV summary:
  Accuracy      : 0.1533 +/- 0.0709
  Balanced-Acc  : 0.3600 +/- 0.0800
  Macro-F1      : 0.1931 +/- 0.0840
  Macro-AUC     : 0.8202 +/- 0.0424
  MCC           : 0.1989 +/- 0.1072


## Step 9 — Ablation

Two families. **Feature arms** ask which node features carry the signal;
**architecture arms** remove one design choice at a time (attention, multi-head
aggregation, class balancing, the superpixel graph itself).

Each arm trains on the same folds with the same budget, so the accuracy delta is
attributable to the removed component.

In [12]:
from cxr_gnn.training.ablation import run_feature_ablation, run_architecture_ablation

abl_feat = run_feature_ablation(orig_graphs, orig_labels, cfg, device, wd)
abl_arch = run_architecture_ablation(orig_graphs, orig_labels, cfg, device, wd,
                                     path_items=orig_items)

viz.plot_ablation(abl_feat, wd, "fig8_ablation_features.png")
viz.plot_ablation(abl_arch, wd, "fig8b_ablation_architecture.png")

for path in [f"{wd}/fig8_ablation_features.png", f"{wd}/fig8b_ablation_architecture.png"]:
    plt.figure(figsize=(14, 5))
    plt.imshow(mpimg.imread(path))
    plt.axis("off")
    plt.show()

2026-07-25 07:56:45 [INFO] cxr_gnn.training.ablation: Running feature ablation (5-fold) ...


2026-07-25 07:56:49 [INFO] cxr_gnn.training.ablation:   Hand-only (12d) +edge            acc 0.278+/-0.046 | F1 0.122+/-0.079 | AUC 0.687+/-0.034


2026-07-25 07:56:54 [INFO] cxr_gnn.training.ablation:   Deep-only (128d) +edge           acc 0.168+/-0.059 | F1 0.160+/-0.068 | AUC 0.820+/-0.066


2026-07-25 07:56:59 [INFO] cxr_gnn.training.ablation:   Hybrid (140d) +edge              acc 0.153+/-0.071 | F1 0.193+/-0.084 | AUC 0.820+/-0.042


2026-07-25 07:57:04 [INFO] cxr_gnn.training.ablation:   Hybrid (140d) no-edge            acc 0.168+/-0.059 | F1 0.234+/-0.012 | AUC 0.799+/-0.082


...

2026-07-25 07:57:27 [WARNING] cxr_gnn.models.cnn_baseline: Pretrained weights unavailable (<urlopen error Tunnel connection failed: 403 Forbidden>). Using random init.


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


2026-07-25 07:57:30 [WARNING] cxr_gnn.models.cnn_baseline: Pretrained weights unavailable (<urlopen error Tunnel connection failed: 403 Forbidden>). Using random init.


2026-07-25 07:57:33 [INFO] cxr_gnn.training.ablation:   - Superpixel graph (CNN only)      acc 0.318 | F1 0.165


2026-07-25 07:57:33 [INFO] cxr_gnn.training.ablation: Saved ablation_architecture.json


## Step 10 — Baselines and the statistical suite

GCN, GraphSAGE, GAT and an **end-to-end fine-tuned ResNet18** on raw pixels.
The fold indices are computed once and shared by all five models, which is what
makes the paired McNemar and DeLong tests valid.

Computed here: MCC, Cohen's kappa, Youden's J, Brier, ECE, log loss (each with
bootstrap CIs), per-class Sens/Spec/PPV/NPV with Wilson CIs, **all** pairwise
DeLong and McNemar comparisons under Holm and BH correction, and the cost
columns (trainable parameters, training time, inference latency) measured rather
than asserted.

In [13]:
from cxr_gnn.training.crossval import run_model_comparison

bl_results = run_model_comparison(orig_graphs, orig_labels, cfg, device, wd,
                                  path_items=orig_items,
                                  include_cnn_baseline=cfg.include_cnn_baseline)
viz.plot_model_comparison(bl_results, wd)

plt.figure(figsize=(14, 5))
plt.imshow(mpimg.imread(f"{wd}/fig9_model_comparison.png"))
plt.axis("off")
plt.show()

with open(f"{wd}/statistical_robustness.json") as f:
    stat_summary = json.load(f)

print(f"\nPairwise comparisons (Holm-corrected), primary = {stat_summary['primary_model']}:")
for r in stat_summary["pairwise_delong_mcnemar"]:
    verdict = "SIGNIFICANT" if r["significant_holm_alpha_0.05"] else "n.s."
    print(f"  {r['model_a']:22s} vs {r['model_b']:22s} {r['metric']:10s} "
          f"p={r['p_value']:.3g} holm={r['holm_corrected_p']:.3g}  {verdict}")

2026-07-25 07:57:36 [INFO] cxr_gnn.training.crossval:   GCN                   : acc 0.362 | F1 0.328 | AUC 0.836 | 41997 params | 0.4 s/fold | 1.7 ms/graph


2026-07-25 07:57:38 [INFO] cxr_gnn.training.crossval:   GraphSAGE             : acc 0.168 | F1 0.211 | AUC 0.851 | 78093 params | 0.4 s/fold | 1.3 ms/graph


2026-07-25 07:57:41 [INFO] cxr_gnn.training.crossval:   GAT                   : acc 0.084 | F1 0.031 | AUC 0.692 | 43197 params | 0.7 s/fold | 2.2 ms/graph


2026-07-25 07:57:46 [INFO] cxr_gnn.training.crossval:   GATv2 Hybrid (ours)   : acc 0.153 | F1 0.193 | AUC 0.820 | 79293 params | 0.8 s/fold | 2.4 ms/graph


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


...

2026-07-25 07:58:08 [INFO] cxr_gnn.training.crossval:   GAT                    vs ResNet18 (CNN baseline) Accuracy   p=0.00151 holm-p=0.0182 SIGNIFICANT


2026-07-25 07:58:08 [INFO] cxr_gnn.training.crossval:   GATv2 Hybrid (ours)    vs ResNet18 (CNN baseline) Macro-AUC  p=0.000768 holm-p=0.00999 SIGNIFICANT


2026-07-25 07:58:08 [INFO] cxr_gnn.training.crossval:   GATv2 Hybrid (ours)    vs ResNet18 (CNN baseline) Accuracy   p=0.0357 holm-p=0.286 n.s.


2026-07-25 07:58:08 [INFO] cxr_gnn.training.crossval: Saved baseline_results.json and statistical_robustness.json



Pairwise comparisons (Holm-corrected), primary = GATv2 Hybrid (ours):
  GCN                    vs GraphSAGE              Macro-AUC  p=0.0621 holm=0.43  n.s.
  GCN                    vs GraphSAGE              Accuracy   p=0.00434 holm=0.0434  SIGNIFICANT
  GCN                    vs GAT                    Macro-AUC  p=9.04e-08 holm=1.72e-06  SIGNIFICANT
  GCN                    vs GAT                    Accuracy   p=1.91e-06 holm=3.24e-05  SIGNIFICANT
  GCN                    vs GATv2 Hybrid (ours)    Macro-AUC  p=0.0649 holm=0.43  n.s.
  GCN                    vs GATv2 Hybrid (ours)    Accuracy   p=0.0026 holm=0.0286  SIGNIFICANT
  GCN                    vs ResNet18 (CNN baseline) Macro-AUC  p=8.63e-05 holm=0.00121  SIGNIFICANT
  GCN                    vs ResNet18 (CNN baseline) Accuracy   p=0.771 holm=1  n.s.
  GraphSAGE              vs GAT                    Macro-AUC  p=2.79e-11 holm=5.59e-10  SIGNIFICANT
  GraphSAGE              vs GAT                    Accuracy   p=0.0312 holm=0.

## Step 11 — Split-ratio sensitivity, and error analysis

Five split ratios × several seeds, reusing the pre-built graph pool: only the
index partition changes, so nothing is re-processed through SLIC. This is what
shows the headline number is not an artefact of one lucky split.

The seed varies **model initialisation as well as the partition** — varying only
the partition leaves every "seed" sharing one initialisation.

The pooled predictions from the best config then feed the per-class table, the
confusion matrix and the error taxonomy, so all three agree by construction.

In [14]:
from cxr_gnn.training.split_sensitivity import run_split_sensitivity
from cxr_gnn.evaluation.error_analysis import run_error_analysis

sens = run_split_sensitivity(all_graphs, all_labels, cfg, device, wd,
                             seeds=cfg.sensitivity_seeds,
                             epochs=cfg.sensitivity_epochs,
                             patience=cfg.sensitivity_patience,
                             n_boot=cfg.n_bootstrap)

print(f"\nBest split configuration: {sens['best_split_config']}")
for cid, ci in sens["bootstrap_ci_by_split"].items():
    print(f"  {cid}: Acc {ci['accuracy_ci']} | Macro-F1 {ci['macro_f1_ci']} | MCC {ci['mcc_ci']}")

pooled = sens["pooled_predictions_best_split"]
err = run_error_analysis(np.array(pooled["y"]), np.array(pooled["pred"]),
                         np.array(pooled["confidence"]), wd)
viz.plot_confusion(np.array(pooled["y"]), np.array(pooled["pred"]), TARGET_NAMES, wd,
                   fname="fig2b_confusion_best_split.png")

2026-07-25 07:58:09 [INFO] cxr_gnn.training.split_sensitivity: Split-ratio sensitivity: 5 configs x 2 seeds = 10 training runs ...


2026-07-25 07:58:10 [INFO] cxr_gnn.training.split_sensitivity:   [S1 seed=42] acc=0.1111 bal-acc=0.2000 macroF1=0.0400 macroAUC=0.7619 mcc=0.0000 n(tr/va/te)=63/9/9


2026-07-25 07:58:10 [INFO] cxr_gnn.training.split_sensitivity:   [S1 seed=43] acc=0.1111 bal-acc=0.2000 macroF1=0.0400 macroAUC=0.7746 mcc=0.0000 n(tr/va/te)=63/9/9


2026-07-25 07:58:11 [INFO] cxr_gnn.training.split_sensitivity:   [S2 seed=42] acc=0.0833 bal-acc=0.2000 macroF1=0.0308 macroAUC=0.8657 mcc=0.0000 n(tr/va/te)=57/12/12


2026-07-25 07:58:12 [INFO] cxr_gnn.training.split_sensitivity:   [S2 seed=43] acc=0.0833 bal-acc=0.2000 macroF1=0.0308 macroAUC=0.8296 mcc=0.0000 n(tr/va/te)=57/12/12


...

2026-07-25 07:58:30 [INFO] cxr_gnn.evaluation.error_analysis:   Tuberculosis (TB) -> Pleural Pathology     n=  4 (25.0% of errors) mean-conf 0.273


2026-07-25 07:58:30 [INFO] cxr_gnn.evaluation.error_analysis:   Chronic Lung Disease -> Pleural Pathology  n=  2 (12.5% of errors) mean-conf 0.265


2026-07-25 07:58:30 [INFO] cxr_gnn.evaluation.error_analysis:   Abstaining below confidence 0.55 would defer 100.0% of errors (and 100.0% of correct predictions).


2026-07-25 07:58:30 [INFO] cxr_gnn.evaluation.error_analysis: Saved outputs/error_analysis.json


'outputs/fig2b_confusion_best_split.png'

## Step 12 — Conformal prediction

Prediction *sets* with a coverage guarantee. A non-singleton set is the model
saying it cannot separate those classes for this image — exactly the
"indeterminate result" a reporting checklist asks about.

In [15]:
from cxr_gnn.evaluation.conformal import run_conformal
from cxr_gnn.training.crossval import _train_one_fold

def train_fold_fn(tr_g, va_g):
    m, _ = _train_one_fold(tr_g, va_g, nfd, efd, cfg, device,
                           epochs=cfg.cv_epochs, patience=cfg.cv_patience)
    return m

conf_results = run_conformal(orig_graphs, orig_labels, train_fold_fn, cfg, device, wd)

print("\nConformal summary (5-fold average):")
for m in ["lac", "aps", "raps", "cclac"]:
    print(f"  {m.upper():6s}: coverage={np.mean(conf_results[m]['cov']):.3f} "
          f"| avg set size={np.mean(conf_results[m]['size']):.2f} "
          f"| singletons={np.mean(conf_results[m]['sing']):.1%}")

2026-07-25 07:58:30 [INFO] cxr_gnn.evaluation.conformal: Conformal prediction (target coverage 90%) ...


2026-07-25 07:58:31 [WARNING] cxr_gnn.training.crossval: [conformal cal/test] Cannot stratify a split of 15 items (smallest class has 1 member(s)); falling back to an unstratified split.


2026-07-25 07:58:31 [INFO] cxr_gnn.evaluation.conformal: Fold 1: LAC 1.00/4.50 | APS 1.00/5.00 | RAPS 1.00/5.00 | CCLAC 0.88/2.38  (cov/size)


2026-07-25 07:58:32 [WARNING] cxr_gnn.training.crossval: [conformal cal/test] Cannot stratify a split of 15 items (smallest class has 1 member(s)); falling back to an unstratified split.


2026-07-25 07:58:32 [INFO] cxr_gnn.evaluation.conformal: Fold 2: LAC 1.00/4.25 | APS 1.00/5.00 | RAPS 1.00/5.00 | CCLAC 0.62/2.88  (cov/size)


...

2026-07-25 07:58:34 [INFO] cxr_gnn.evaluation.conformal:   RAPS  : cov 1.000+/-0.000 | size 5.00 | singletons 0.0%


2026-07-25 07:58:34 [INFO] cxr_gnn.evaluation.conformal:   CCLAC : cov 0.700+/-0.164 | size 1.99 | singletons 31.1%


2026-07-25 07:58:34 [INFO] cxr_gnn.evaluation.conformal: -> Recommended: RAPS (best size/coverage trade-off)


2026-07-25 07:58:34 [INFO] cxr_gnn.evaluation.conformal: Saved outputs/conformal_results.json



Conformal summary (5-fold average):
  LAC   : coverage=0.943 | avg set size=4.29 | singletons=0.0%
  APS   : coverage=1.000 | avg set size=5.00 | singletons=0.0%
  RAPS  : coverage=1.000 | avg set size=5.00 | singletons=0.0%
  CCLAC : coverage=0.700 | avg set size=1.99 | singletons=31.1%


## Step 13 — Calibration

ECE below 0.05 reads as well calibrated; above 0.10 the probabilities should not
be quoted without correction. When that happens, a temperature is fitted on one
half of the pooled out-of-fold predictions and evaluated on the other, so the
post-scaling number is not fitted on the data it is reported on.

In [16]:
from cxr_gnn.evaluation.calibration import compute_calibration

cal = compute_calibration(orig_graphs, orig_labels, train_fold_fn, cfg, device, wd)
viz.plot_calibration(cal, wd)

print(f"ECE {cal['ECE']:.4f} | MCE {cal['MCE']:.4f} | "
      f"OOF acc {cal['oof_accuracy']:.4f} | mean confidence {cal['avg_confidence']:.4f}")
if cal["temperature_scaling"]:
    ts = cal["temperature_scaling"]
    print(f"Temperature scaling: T={ts['temperature']:.3f}, "
          f"ECE {ts['ece_before_on_eval_half']:.4f} -> {ts['ece_after']:.4f} (held-out half)")

plt.figure(figsize=(13, 5))
plt.imshow(mpimg.imread(f"{wd}/fig10_calibration.png"))
plt.axis("off")
plt.show()

2026-07-25 07:58:38 [INFO] cxr_gnn.evaluation.calibration: ECE: 0.2331 | MCE: 0.6797 | OOF acc: 0.1528


2026-07-25 07:58:38 [WARNING] cxr_gnn.evaluation.calibration: ECE=0.233 > 0.10 -- fitting temperature scaling.


2026-07-25 07:58:38 [INFO] cxr_gnn.evaluation.calibration: Temperature scaling: T=515741.659, ECE 0.2207 -> 0.0333 on the held-out half


2026-07-25 07:58:38 [INFO] cxr_gnn.evaluation.calibration: Saved outputs/calibration.json


ECE 0.2331 | MCE 0.6797 | OOF acc 0.1528 | mean confidence 0.2726
Temperature scaling: T=515741.659, ECE 0.2207 -> 0.0333 (held-out half)


## Step 14 — Robustness under input perturbation

Each perturbation is applied to the test images and the graphs are rebuilt
through the identical SLIC + encoder path, so what is measured is the whole
pipeline's sensitivity, not just the classifier's.

In [17]:
from cxr_gnn.evaluation.robustness import run_robustness_study

rob = run_robustness_study(model, kept_items["test"], cfg, encoder, device, wd)
viz.plot_robustness(rob, wd)

plt.figure(figsize=(12, 5))
plt.imshow(mpimg.imread(f"{wd}/fig11_robustness.png"))
plt.axis("off")
plt.show()

2026-07-25 07:58:39 [INFO] cxr_gnn.evaluation.robustness: Robustness study: clean reference + 6 perturbations ...


2026-07-25 07:58:41 [INFO] cxr_gnn.evaluation.robustness:   None (clean test set)          acc 0.4444 (reference)


2026-07-25 07:58:43 [INFO] cxr_gnn.evaluation.robustness:   Gaussian noise (sigma=0.02)    acc 0.4444 (delta +0.0000, retention 100.0%, grade A)


2026-07-25 07:58:45 [INFO] cxr_gnn.evaluation.robustness:   Gaussian noise (sigma=0.05)    acc 0.2222 (delta -0.2222, retention 50.0%, grade D)


2026-07-25 07:58:47 [INFO] cxr_gnn.evaluation.robustness:   JPEG compression (q=50)        acc 0.4444 (delta +0.0000, retention 100.0%, grade A)


2026-07-25 07:58:49 [INFO] cxr_gnn.evaluation.robustness:   Rotation (+/-10 deg)           acc 0.4444 (delta +0.0000, retention 100.0%, grade A)


2026-07-25 07:58:51 [INFO] cxr_gnn.evaluation.robustness:   Brightness shift (+/-20%)      acc 0.1111 (delta -0.3333, retention 25.0%, grade D)


2026-07-25 07:58:53 [INFO] cxr_gnn.evaluation.robustness:   Contrast reduction (-30%)      acc 0.6667 (delta +0.2222, retention 150.0%, grade A)


2026-07-25 07:58:53 [INFO] cxr_gnn.evaluation.robustness: Saved outputs/robustness.json


## Step 15 — Evaluation tables

Every table is rebuilt from the JSON artifacts written above, so no number in
the write-up is transcribed by hand. A table whose artifact is missing prints
"not available — run step X" rather than a plausible-looking placeholder.

In [18]:
from IPython.display import Markdown, display
from cxr_gnn.evaluation.tables import build_all_tables

display(Markdown(build_all_tables(wd)))

2026-07-25 07:58:53 [INFO] cxr_gnn.evaluation.tables: Saved outputs/evaluation_tables.md


**Table 1. Split configurations, aggregate performance (mean +/- SD over 2 seeds) and bootstrapped 95% CIs.**

| Config | Ratio (Tr/Va/Te) | n (Tr/Va/Te) | Accuracy | Bal. Acc. | Macro F1 | Macro AUC | Accuracy 95% CI | Macro F1 95% CI | Macro AUC 95% CI | MCC 95% CI |
|---|---|---|---|---|---|---|---|---|---|---|
| S1 | 80%/10%/10% | 63 / 9 / 9 | 0.1111 +/- 0.0000 | 0.2000 +/- 0.0000 | 0.0400 +/- 0.0000 | 0.7683 +/- 0.0063 | 0.111 (0.000-0.278) | 0.040 (0.000-0.093) | 0.735 (0.548-0.855) | 0.000 (0.000-0.000) |
| S2 | 70%/15%/15% | 57 / 12 / 12 | 0.0833 +/- 0.0000 | 0.2000 +/- 0.0000 | 0.0308 +/- 0.0000 | 0.8477 +/- 0.0181 | 0.083 (0.000-0.208) | 0.031 (0.000-0.070) | 0.759 (0.610-0.871) | 0.000 (0.000-0.000) |
| S3 | 75%/15%/10% | 60 / 12 / 9 | 0.1111 +/- 0.0000 | 0.2000 +/- 0.0000 | 0.0400 +/- 0.0000 | 0.7825 +/- 0.0254 | 0.111 (0.000-0.278) | 0.040 (0.000-0.093) | 0.724 (0.535-0.853) | 0.000 (0.000-0.000) |
| S4 | 70%/20%/10% | 57 / 15 / 9 | 0.1111 +/- 0.0000 | 0.2000 +/- 0.0000 | 0.0400 +/- 0.0000 | 0.8413 +/- 0.0016 | 0.111 (0.000-0.278) | 0.040 (0.000-0.093) | 0.756 (0.561-0.888) | 0.000 (0.000-0.000) |
| S5 | 60%/20%/20% | 51 / 15 / 15 | 0.0667 +/- 0.0000 | 0.2000 +/- 0.0000 | 0.0250 +/- 0.0000 | 0.8313 +/- 0.0542 | 0.067 (0.000-0.167) | 0.025 (0.000-0.059) | 0.693 (0.548-0.825) | 0.000 (0.000-0.000) |
| **Mean** | - | 81 total | 0.0967 +/- 0.0185 | 0.2000 +/- 0.0000 | 0.0352 +/- 0.0062 | 0.8142 +/- 0.0324 | - | - | - | - |

All splits are stratified by class and performed on raw images before augmentation (no image-level leakage). Seeds 42-43. Best configuration by mean accuracy: **S1**; bootstrap B = 300.

---

**Table 2. Stratified 5-fold cross-validation.**

| Fold | Train n | Val n | Test n | Accuracy | Balanced Acc. | Macro F1 | Macro AUC | MCC |
|---|---|---|---|---|---|---|---|---|
| 1 | 48 | 9 | 15 | 0.1333 | 0.4000 | 0.2267 | 0.8149 | 0.2200 |
| 2 | 48 | 9 | 15 | 0.1333 | 0.4000 | 0.2267 | 0.7713 | 0.2200 |
| 3 | 49 | 9 | 14 | 0.1429 | 0.4000 | 0.2286 | 0.8533 | 0.2288 |
| 4 | 49 | 9 | 14 | 0.2857 | 0.4000 | 0.2571 | 0.7795 | 0.3257 |
| 5 | 49 | 9 | 14 | 0.0714 | 0.2000 | 0.0267 | 0.8818 | 0.0000 |
| **Mean +/- SD** | - | - | - | 0.1533 +/- 0.0709 | 0.3600 +/- 0.0800 | 0.1931 +/- 0.0840 | 0.8202 +/- 0.0424 | 0.1989 +/- 0.1072 |

Folds are stratified on the original (non-augmented) images only; augmented graphs never enter a held-out fold.

---

**Table 3. Per-class performance, best split configuration (S1), pooled over 2 seeds. Wilson score 95% CIs.**

| Class | Support | Prevalence | Sensitivity (95% CI) | Specificity (95% CI) | PPV (95% CI) | NPV (95% CI) | F1 | AUC | LR+ | LR- |
|---|---|---|---|---|---|---|---|---|---|---|
| Cardiac Pathology | 4 | 22.2% | 0.000 (0.000-0.490) | 1.000 (0.785-1.000) | n/a (n/a-n/a) | 0.778 (0.548-0.910) | 0.000 | 0.411 | inf | 1.00 |
| Chronic Lung Disease | 2 | 11.1% | 0.000 (0.000-0.658) | 1.000 (0.806-1.000) | n/a (n/a-n/a) | 0.889 (0.672-0.969) | 0.000 | 0.750 | inf | 1.00 |
| Normal | 6 | 33.3% | 0.000 (0.000-0.390) | 1.000 (0.758-1.000) | n/a (n/a-n/a) | 0.667 (0.437-0.837) | 0.000 | 0.819 | inf | 1.00 |
| Pleural Pathology | 2 | 11.1% | 1.000 (0.342-1.000) | 0.000 (0.000-0.194) | 0.111 (0.031-0.328) | n/a (n/a-n/a) | 0.200 | 1.000 | 1.00 | n/a |
| Tuberculosis (TB) | 4 | 22.2% | 0.000 (0.000-0.490) | 1.000 (0.785-1.000) | n/a (n/a-n/a) | 0.778 (0.548-0.910) | 0.000 | 0.696 | inf | 1.00 |
| **Macro average** | 18 | 100% | 0.200 | 0.800 | 0.111 | 0.778 | 0.040 | 0.735 | 1.00 | 1.00 |

Wilson score intervals are used for every proportion: more reliable than the normal approximation at the sample sizes of the minority classes. Sensitivity here is the same quantity as per-class recall in Table 4 and is computed from the same pooled predictions, so the two tables agree by construction.

---

**Table 4. Confusion matrix, best split configuration (S1), pooled over seeds (rows = true, columns = predicted).**

| True \ Predicted | Cardiac Pathology | Chronic Lung Disease | Normal | Pleural Pathology | Tuberculosis (TB) | Total | Recall |
|---|---|---|---|---|---|---|---|
| Cardiac Pathology | 0 | 0 | 0 | 4 | 0 | 4 | 0.000 |
| Chronic Lung Disease | 0 | 0 | 0 | 2 | 0 | 2 | 0.000 |
| Normal | 0 | 0 | 0 | 6 | 0 | 6 | 0.000 |
| Pleural Pathology | 0 | 0 | 0 | 2 | 0 | 2 | 1.000 |
| Tuberculosis (TB) | 0 | 0 | 0 | 4 | 0 | 4 | 0.000 |
| **Total predicted** | 0 | 0 | 0 | 18 | 0 | 18 | - |
| **Precision** | n/a | n/a | n/a | 0.111 | n/a | - | 0.111 (accuracy) |

---

**Table 5. Statistical significance, all pairwise comparisons.**

| Family | Comparison | Metric | Test | Statistic | p-value | Holm-corrected p | Delta (A-B) | Effect size | Significant (alpha=.05) |
|---|---|---|---|---|---|---|---|---|---|
| Split-ratio | S1 vs S2 | accuracy | Paired t-test | t = inf | 0 | 0 | +0.0278 | d = 0.00 (negligible) | Yes |
| Split-ratio | S1 vs S2 | macro_f1 | Paired t-test | t = inf | 0 | 0 | +0.0092 | d = 0.00 (negligible) | Yes |
| Split-ratio | S1 vs S3 | accuracy | Paired t-test | t = 0.00 | 1 | 1 | +0.0000 | d = 0.00 (negligible) | No |
| Split-ratio | S1 vs S4 | accuracy | Paired t-test | t = 0.00 | 1 | 1 | +0.0000 | d = 0.00 (negligible) | No |
| Split-ratio | S1 vs S5 | accuracy | Paired t-test | t = inf | 0 | 0 | +0.0444 | d = 0.00 (negligible) | Yes |
| Split-ratio | S2 vs S4 | accuracy | Paired t-test | t = -inf | 0 | 0 | -0.0278 | d = 0.00 (negligible) | Yes |
| Model-vs-model | GCN vs GraphSAGE | Macro-AUC | DeLong (one-vs-rest, Fisher-combined) | z = -2.44 | 0.0621 | 0.43 | -0.0356 | - | No |
| Model-vs-model | GCN vs GraphSAGE | Accuracy | McNemar (exact) | chi2 = 7.68 | 0.00434 | 0.0434 | +0.1944 | - | Yes (GCN better) |
| Model-vs-model | GCN vs GAT | Macro-AUC | DeLong (one-vs-rest, Fisher-combined) | z = 4.48 | 9.04e-08 | 1.72e-06 | +0.0810 | - | Yes (GCN better) |
| Model-vs-model | GCN vs GAT | Accuracy | McNemar (exact) | chi2 = 18.05 | 1.91e-06 | 3.24e-05 | +0.2778 | - | Yes (GCN better) |
| Model-vs-model | GCN vs GATv2 Hybrid (ours) | Macro-AUC | DeLong (one-vs-rest, Fisher-combined) | z = -1.14 | 0.0649 | 0.43 | +0.0022 | - | No |
| Model-vs-model | GCN vs GATv2 Hybrid (ours) | Accuracy | McNemar (exact) | chi2 = 8.52 | 0.0026 | 0.0286 | +0.2083 | - | Yes (GCN better) |
| Model-vs-model | GCN vs ResNet18 (CNN baseline) | Macro-AUC | DeLong (one-vs-rest, Fisher-combined) | z = 3.84 | 8.63e-05 | 0.00121 | +0.1768 | - | Yes (GCN better) |
| Model-vs-model | GCN vs ResNet18 (CNN baseline) | Accuracy | McNemar (exact) | chi2 = 0.09 | 0.771 | 1 | +0.0417 | - | No |
| Model-vs-model | GraphSAGE vs GAT | Macro-AUC | DeLong (one-vs-rest, Fisher-combined) | z = 5.24 | 2.79e-11 | 5.59e-10 | +0.1167 | - | Yes (GraphSAGE better) |
| Model-vs-model | GraphSAGE vs GAT | Accuracy | McNemar (exact) | chi2 = 4.17 | 0.0312 | 0.281 | +0.0833 | - | No |
| Model-vs-model | GraphSAGE vs GATv2 Hybrid (ours) | Macro-AUC | DeLong (one-vs-rest, Fisher-combined) | z = 1.94 | 0.327 | 0.98 | +0.0379 | - | No |
| Model-vs-model | GraphSAGE vs GATv2 Hybrid (ours) | Accuracy | McNemar (exact) | chi2 = 0.00 | 1 | 1 | +0.0139 | - | No |
| Model-vs-model | GraphSAGE vs ResNet18 (CNN baseline) | Macro-AUC | DeLong (one-vs-rest, Fisher-combined) | z = 4.78 | 1.58e-05 | 0.000237 | +0.2124 | - | Yes (GraphSAGE better) |
| Model-vs-model | GraphSAGE vs ResNet18 (CNN baseline) | Accuracy | McNemar (exact) | chi2 = 3.45 | 0.0614 | 0.43 | -0.1528 | - | No |
| Model-vs-model | GAT vs GATv2 Hybrid (ours) | Macro-AUC | DeLong (one-vs-rest, Fisher-combined) | z = -3.15 | 3.24e-06 | 5.19e-05 | -0.0788 | - | Yes (GATv2 Hybrid (ours) better) |
| Model-vs-model | GAT vs GATv2 Hybrid (ours) | Accuracy | McNemar (exact) | chi2 = 2.29 | 0.125 | 0.5 | -0.0694 | - | No |
| Model-vs-model | GAT vs ResNet18 (CNN baseline) | Macro-AUC | DeLong (one-vs-rest, Fisher-combined) | z = 2.16 | 5.73e-07 | 1.03e-05 | +0.0958 | - | Yes (GAT better) |
| Model-vs-model | GAT vs ResNet18 (CNN baseline) | Accuracy | McNemar (exact) | chi2 = 9.48 | 0.00151 | 0.0182 | -0.2361 | - | Yes (ResNet18 (CNN baseline) better) |
| Model-vs-model | GATv2 Hybrid (ours) vs ResNet18 (CNN baseline) | Macro-AUC | DeLong (one-vs-rest, Fisher-combined) | z = 3.48 | 0.000768 | 0.00999 | +0.1746 | - | Yes (GATv2 Hybrid (ours) better) |
| Model-vs-model | GATv2 Hybrid (ours) vs ResNet18 (CNN baseline) | Accuracy | McNemar (exact) | chi2 = 4.32 | 0.0357 | 0.286 | -0.1667 | - | No |

Holm-Bonferroni correction is applied within each family. The DeLong row reports the Stouffer-combined z across one-vs-rest classes (signed, so the direction is readable) while the p-value is Fisher-combined; McNemar is the exact paired test on the same held-out instances.

---

**Table 6. Model comparison: discrimination, agreement, calibration and cost.**

| Model | Acc. | Macro F1 | Macro AUC | Bal. Acc. | MCC (95% CI) | Cohen's kappa (95% CI) | Youden J | Brier | ECE | Log loss | Trainable params | Train (min/fold) | Infer. (ms) |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| GCN | 0.361 | 0.388 | 0.816 | 0.467 | 0.333 (0.221-0.433) | 0.169 (0.074-0.281) | 0.296 | 0.768 | 0.103 | 1.533 | 41,997 | 0.01 | 1.7 |
| GraphSAGE | 0.167 | 0.195 | 0.852 | 0.400 | 0.206 (0.094-0.299) | 0.091 (0.031-0.158) | 0.218 | 0.810 | 0.198 | 1.649 | 78,093 | 0.01 | 1.3 |
| GAT | 0.083 | 0.031 | 0.735 | 0.200 | 0.000 (0.000-0.000) | 0.000 (0.000-0.000) | 0.000 | 0.824 | 0.256 | 1.671 | 43,197 | 0.01 | 2.2 |
| GATv2 Hybrid (ours) | 0.153 | 0.215 | 0.814 | 0.367 | 0.223 (0.106-0.295) | 0.076 (0.016-0.135) | 0.182 | 0.827 | 0.233 | 1.686 | 79,293 | 0.01 | 2.4 |
| ResNet18 (CNN baseline) | 0.319 | 0.185 | 0.639 | 0.267 | 0.094 (-0.015-0.214) | 0.071 (-0.011-0.161) | 0.081 | 0.790 | 0.094 | 1.585 | 11,179,077 | 0.05 | 10.3 |

MCC and Cohen's kappa stay meaningful under class imbalance, unlike raw accuracy. Brier score, ECE and log loss quantify how trustworthy the probabilities are, which any claim of clinical usefulness depends on. Parameter counts are TRAINABLE parameters: the GATv2 hybrid's ResNet18 encoder is frozen and therefore excluded, which is why its count is far below the fine-tuned CNN baseline's. Timings are for the hardware this run used and are not comparable across machines.

---

**Table 7a. Ablation study: contribution of each component.**

| Configuration | Accuracy | Macro F1 | Macro AUC | MCC | Delta Acc. vs reference | Trainable params | Infer. (ms) |
|---|---|---|---|---|---|---|---|
| Full GATv2 Hybrid (reference) | 0.1533 | 0.1931 | 0.8202 | 0.1989 | - | 79,293 | 2.5 |
| - GATv2 -> GCN layer | 0.3619 | 0.3278 | 0.8355 | 0.3166 | +0.2086 | 41,997 | 1.7 |
| - Multi-head attention (1 head) | 0.1248 | 0.1031 | 0.6977 | 0.0923 | -0.0286 | 23,997 | 2.2 |
| - Class weighting / sampler | 0.3457 | 0.1550 | 0.5424 | 0.1713 | +0.1924 | 79,293 | 2.8 |
| - Superpixel graph (CNN only) | 0.3181 | 0.1650 | 0.7481 | 0.1598 | +0.1648 | 11,179,077 | n/a |
| Hand-only (12d) +edge | 0.2781 | 0.1216 | 0.6873 | 0.0698 | +0.1248 | 29,885 | 2.4 |
| Deep-only (128d) +edge | 0.1676 | 0.1600 | 0.8202 | 0.1816 | +0.0143 | 74,661 | 2.5 |
| Hybrid (140d) +edge | 0.1533 | 0.1931 | 0.8202 | 0.1989 | +0.0000 | 79,293 | 2.7 |
| Hybrid (140d) no-edge | 0.1676 | 0.2335 | 0.7985 | 0.2447 | +0.0143 | 78,813 | 2.2 |

Each arm removes exactly one component and is trained on the same folds with the same budget, so the delta is attributable to that component. A positive delta means the removed component was not helping.

---

**Table 7b. Robustness: performance under input perturbation.**

| Perturbation | Accuracy | Delta Acc. | Macro F1 | Macro AUC | MCC | Retention | Grade |
|---|---|---|---|---|---|---|---|
| None (clean test set) | 0.4444 | - | 0.4889 | 0.9429 | 0.4637 | 100.0% | Reference |
| Gaussian noise (sigma=0.02) | 0.4444 | +0.0000 | 0.4889 | 0.9429 | 0.4637 | 100.0% | A |
| Gaussian noise (sigma=0.05) | 0.2222 | -0.2222 | 0.2571 | 1.0000 | 0.1406 | 50.0% | D |
| JPEG compression (q=50) | 0.4444 | +0.0000 | 0.4889 | 0.9286 | 0.4637 | 100.0% | A |
| Rotation (+/-10 deg) | 0.4444 | +0.0000 | 0.5000 | 0.8952 | 0.3723 | 100.0% | A |
| Brightness shift (+/-20%) | 0.1111 | -0.3333 | 0.0400 | 0.7663 | 0.0000 | 25.0% | D |
| Contrast reduction (-30%) | 0.6667 | +0.2222 | 0.5100 | 0.9429 | 0.5992 | 150.0% | A |

Retention = perturbed accuracy / clean accuracy, on 9 held-out test images. Grades: A >= 96%, B >= 92%, C >= 88%, D below. Perturbations are applied to the image and the graph is rebuilt through the identical SLIC + encoder path.

---

**Table 7c. Error analysis: misclassification taxonomy.**

| Error pattern | Count | % of errors | Mean confidence |
|---|---|---|---|
| Normal -> Pleural Pathology | 6 | 37.5% | 0.274 |
| Cardiac Pathology -> Pleural Pathology | 4 | 25.0% | 0.300 |
| Tuberculosis (TB) -> Pleural Pathology | 4 | 25.0% | 0.273 |
| Chronic Lung Disease -> Pleural Pathology | 2 | 12.5% | 0.265 |
| **Total errors** | 16 | 100% | 0.279 |

Errors are 88.9% of 18 pooled predictions. Mean confidence on correct predictions is 0.337 vs 0.279 on errors. Abstaining below confidence 0.55 would defer 100.0% of errors, at the cost of deferring 100.0% of correct predictions.

---

**Table 8. Literature benchmarking (qualitative positioning).**

| Study / Model | Year | Dataset (n) | Task / Classes | Approach | Acc. | Macro AUC | Macro F1 | Sens. (macro) |
|---|---|---|---|---|---|---|---|---|
| CheXNet (Rajpurkar et al., 2017) | 2017 | ChestX-ray14 (112,120 images) | 14 findings, multi-label binary | 121-layer DenseNet | n/a (multi-label) | ~0.83-0.84 mean AUROC | 0.435 (pneumonia) | n/a |
| COVID-Net (Wang, Lin & Wong, 2020) | 2020 | COVIDx (~13,975 images) | 3-class multiclass | Lightweight CNN (PEPX) | 0.933 | n/a | n/a | ~0.91 (COVID class) |
| MIMIC-CXR single-source benchmark | recent | MIMIC-CXR (~377,110 images) | Multi-label findings | CNN multi-label classifier | n/a | ~0.75 mean AUROC | n/a | n/a |
| **GATv2 Hybrid (this work)** | this study | This dataset (81) | 5-class multiclass | Frozen ResNet18 features + superpixel GATv2 | 0.444 | 0.943 | 0.489 | 0.600 |

*Because label sets, task framing and dataset scale differ by two to three orders of magnitude, this table supports only relative discussion (direction and size of the gap, with the dataset-scale caveat), never a claim of superiority. The only controlled head-to-head comparison in this work is the CNN baseline in Table 6, which is trained and evaluated on identical splits.*

In [19]:
from cxr_gnn.evaluation.reporting import reporting_checklist_markdown

# "addressed" is verified against the artifact actually existing on disk.
display(Markdown(reporting_checklist_markdown(wd)))

### TRIPOD-AI checklist

| Item | Status | Evidence |
|---|---|---|
| Title/abstract identifies the study as developing an AI/ML prediction model | needs manual entry | Add when writing up. |
| Source of data and eligibility criteria described | needs manual entry | Document the source institution(s) and inclusion criteria. |
| Outcome (target classes) clearly defined | addressed | 5-class taxonomy fixed in config.py (CLASS2IDX). |
| Predictors (features) fully specified | addressed | Hand-crafted + deep node features documented in graph.py / encoder.py. |
| Sample size / cases per class justified or acknowledged as a limitation | addressed | `split_sensitivity.json` -- Split-ratio sensitivity plus per-class support. |
| Missing / unusable data handling described | addressed | `run.log` -- Degenerate-segmentation drops are logged per split by cache.py. |
| Development vs validation data separation stated | addressed | `split_sensitivity.json` -- Image-level stratified split before augmentation. |
| Internal validation method reported | addressed | `cv_results.json` -- 5-fold CV plus bootstrap 95% CIs. |
| Performance measures justified for the clinical task and class imbalance | addressed | `statistical_robustness.json` -- Macro-F1, balanced accuracy, MCC, kappa. |
| Calibration reported | addressed | `calibration.json` -- ECE/MCE, reliability diagram, temperature scaling. |
| Model updating / re-calibration discussed | N/A | Out of scope for a retrospective single-institution study. |
| Comparison against existing models / baselines | addressed | `baseline_results.json` -- GCN/GraphSAGE/GAT/ResNet18 plus literature table. |
| Uncertainty quantification for individual predictions | addressed | `conformal_results.json` -- Conformal prediction sets with coverage guarantees. |
| External validation on an independent dataset | N/A | Single-source data only -- stated as a limitation. |
| Code / model availability statement | addressed | This repository; add a DOI at publication time. |

### STARD-AI checklist

| Item | Status | Evidence |
|---|---|---|
| Study design (retrospective / prospective) stated | needs manual entry | State explicitly in Methods. |
| Reference standard (ground-truth labelling process) described | needs manual entry | Document the radiologist/clinical labelling protocol. |
| Flow of images (inclusion, exclusions, degenerate cases) reported | addressed | `run.log` -- Per-split drop counts logged by cache.py. |
| Distribution of severity / alternative diagnoses in the sample | needs manual entry | Add clinical characterisation if available. |
| Statistical methods pre-specified and matched to the data | addressed | `statistical_robustness.json` -- Wilson CIs at small n, non-parametric bootstrap, Holm/FDR correction. |
| Indeterminate results handling | addressed | `conformal_results.json` -- Non-singleton conformal sets flag exactly these. |
| Adverse events / harms of testing discussed | N/A | Not applicable to retrospective image classification. |

## Step 16 — Summary of artifacts

In [20]:
print("=" * 60)
print("RUN SUMMARY")
print("=" * 60)
if USING_SYNTHETIC:
    print("\n*** SYNTHETIC PHANTOM DATA -- smoke test only, not findings. ***")
if FAST:
    print("*** FAST mode -- reduced budgets, numbers are not reportable. ***")

print(f"\n[Test set]  " + "  ".join(f"{k}={v:.4f}" for k, v in test_metrics.items()
                                     if isinstance(v, float)))
print(f"[5-fold CV] accuracy={cv_results['Accuracy']['mean']:.4f} "
      f"+/- {cv_results['Accuracy']['std']:.4f}, MCC={cv_results['MCC']['mean']:.4f}")
print(f"[Best split] {sens['best_split_config']}")
print(f"[Calibration] ECE={cal['ECE']:.4f}")

print("\n[Files written]")
for p in sorted(glob.glob(f"{wd}/*.json") + glob.glob(f"{wd}/*.md")
                + glob.glob(f"{wd}/*.png") + glob.glob(f"{wd}/*.pt")):
    print(f"  {os.path.basename(p):42s} {os.path.getsize(p) / 1024:8.1f} KB")

RUN SUMMARY

*** SYNTHETIC PHANTOM DATA -- smoke test only, not findings. ***
*** FAST mode -- reduced budgets, numbers are not reportable. ***

[Test set]  accuracy=0.4444  balanced_accuracy=0.6000  macro_f1=0.4889  macro_auc=0.9429  mcc=0.4637  train_accuracy_eval_mode=0.5368
[5-fold CV] accuracy=0.1533 +/- 0.0709, MCC=0.1989
[Best split] S1
[Calibration] ECE=0.2331

[Files written]
  ablation_architecture.json                      3.3 KB
  ablation_features.json                          2.7 KB
  baseline_results.json                           2.8 KB
  best_gatv2.pt                                 324.6 KB
  calibration.json                                0.8 KB
  conformal_results.json                          1.2 KB
  cv_results.json                                 2.5 KB
  error_analysis.json                             1.2 KB
  evaluation_tables.md                           14.1 KB
  fig10_calibration.png                          55.9 KB
  fig11_robustness.png                    